# Agent 历史越来越长，必须全部放进 Prompt 吗？

## V0.4 Context VM

这个 lab 让你直接比较 durable Session truth 和 model-visible working set。

**Core:** Truth != Context.

In [ ]:
from pathlib import Path
import sys

def find_repo_root(start=Path.cwd()):
    for path in (start, *start.parents):
        if (path / "agentkernel").is_dir():
            return path
    raise RuntimeError("Run this notebook from inside the AgentKernel repository.")

REPOSITORY_ROOT = find_repo_root()
if str(REPOSITORY_ROOT) not in sys.path:
    sys.path.insert(0, str(REPOSITORY_ROOT))
LABS_ROOT = REPOSITORY_ROOT / "examples" / "labs"
if str(LABS_ROOT) not in sys.path:
    sys.path.insert(0, str(LABS_ROOT))

from lab_helpers import event_rows, grant_rows, print_table, process_row, trajectory

## 1. Create more durable facts than the model budget should carry

In [ ]:
from agentkernel import (
    ApproximateTokenEstimator, ContextBudget, ContextManager,
    ContextProjector, EventType, Session,
)

def append_text_turn(session: Session, turn: int, user_content: str, assistant_content: str) -> None:
    session.append(EventType.TURN_START, {"turn": turn})
    session.append(EventType.USER_MESSAGE, {"turn": turn, "content": user_content})
    session.append(EventType.STEP_START, {"turn": turn, "step": 1})
    session.append(EventType.ASSISTANT_MESSAGE, {"turn": turn, "step": 1, "content": assistant_content, "tool_calls": []})
    session.append(EventType.STEP_END, {"turn": turn, "step": 1, "outcome": "done"})
    session.append(EventType.TURN_END, {"turn": turn, "reason": "completed"})

session = Session("lab-v0-4-session")
for turn in range(1, 8):
    append_text_turn(
        session,
        turn,
        f"user fact {turn}: " + ("durable text " * 14),
        f"assistant response {turn}: " + ("model-visible projection " * 10),
    )
print_table([
    {"fact": "durable events", "value": len(session.events)},
    {"fact": "durable messages", "value": len(session.derive_messages())},
])

## 2. Project Context Pages, then build a bounded Working Set

In [ ]:
projector = ContextProjector(ApproximateTokenEstimator(1))
pages = projector.project(session, system_prompt="Keep the answer concise.")
working_set = ContextManager(projector=projector).build_working_set(
    session,
    current_turn=7,
    budget=ContextBudget(max_tokens=420),
    system_prompt="Keep the answer concise.",
)
print_table([
    {"fact": "projected pages", "value": len(pages)},
    {"fact": "selected pages", "value": working_set.metrics.selected_pages},
    {"fact": "evicted pages", "value": working_set.metrics.evicted_pages},
    {"fact": "selected tokens", "value": working_set.metrics.selected_tokens},
    {"fact": "model messages", "value": len(working_set.to_messages())},
    {"fact": "context equals truth", "value": len(working_set.to_messages()) == len(session.derive_messages())},
])

## 3. Inspect selected vs durable data

In [ ]:
print_table([
    {"view": "durable truth", "count": len(session.derive_messages()), "owned_by": "Session event log"},
    {"view": "model-visible context", "count": len(working_set.to_messages()), "owned_by": "Context VM projection"},
])
trajectory("Session events", "Context pages", "Working set selection", "Model request")

## Invariant

Context is a bounded projection. Durable truth remains in Session events.

## WHAT THIS DEMONSTRATES / 本实验验证什么

- Context VM can select a smaller model-visible working set.
- Eviction from prompt context is not deletion from durable truth.

## WHAT THIS DOES NOT DEMONSTRATE / 本实验不证明什么

- It does not prove semantic summary quality.
- It does not benchmark every context policy.
- It does not use a real model provider.